# Pretraining and Supervised Finetuning

## Libraries

In [ ]:
# Installation d'Unsloth optimisé pour Colab
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
# Installation de vos autres outils
!pip install polars datasets

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-v3gp1hfq/unsloth_70f89f301d92460883d6ebd6b6473e1c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-v3gp1hfq/unsloth_70f89f301d92460883d6ebd6b6473e1c
  Resolved https://github.com/unslothai/unsloth.git to commit 6207fc81630e283fb72960f2117949c2ca89b3ab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 23.6 MB/s eta 0:00:00
   ━━━━

In [ ]:
# Optimized data processing
import polars as pl
from datasets import load_dataset

# Deep learning engine
import torch

# Model and tokenizer import via Unsloth
from unsloth import FastLanguageModel

# Remaining Transformers elements
from transformers import TrainingArguments, AutoTokenizer, DataCollatorForSeq2Seq, Trainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## Loading the model and the tokenizer

In [ ]:
MODEL_NAME = "unsloth/Llama-3.2-1B"
max_seq_length = 2048 # Unsloth handles context size automatically

# Model and tokenizer initialisation
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = torch.float16,
    load_in_4bit = True, # Quantization
)

==((====))==  Unsloth 2026.10.3: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-1B-unsloth-bnb-4bit as a legacy tokenizer.


## Lora

In [ ]:
model = FastLanguageModel.get_peft_model(
    model = model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth recommends 0 for maximum speed
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)



Unsloth 2026.10.3 patched 16 layers with 16 QKV layers, 16 O layers and 16 MLP layers.


## Test the model before finetuning

In [ ]:
generation_config = model.generation_config
generation_config.max_new_tokens = 200
generation_config.temperature = 0.7
generation_config.top_p = 0.7
generation_config.num_return_sequences = 1
generation_config.pad_token_id = tokenizer.eos_token_id
generation_config.eos_token_id = tokenizer.eos_token_id
generation_config.do_sample = True


In [ ]:
prompt = "<human>: What equipment do I need for rock climbing?\n<assistant>:"

device = "cuda:0"

encoding = tokenizer(text=prompt, return_tensors="pt").to(device)
with torch.inference_mode():
  outputs = model.generate(
      input_ids = encoding.input_ids,
      attention_mask = encoding.attention_mask,
      generation_config = generation_config,
  )
print(tokenizer.decode(outputs[0],skip_special_token = True))

Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|begin_of_text|><human>: What equipment do I need for rock climbing?
<assistant>: I don't think you need anything special for rock climbing. If you are new to climbing, I suggest you to buy a good climbing shoes and a good harness. It's a good idea to have a harness and climbing shoes with you, especially if you are going to climb in a new place. If you are a beginner, you should also buy a good rope, which should be at least 50 meters long. You can also buy a rope with a carabiner, which is a metal hook, which you can use to attach your climbing shoes and harness. If you are a beginner, you should also buy a good rope, which should be at least 50 meters long. You can also buy a rope with a carabiner, which is a metal hook, which you can use to attach your climbing shoes and harness. If you are a beginner, you should also buy a good rope, which should be at least 50 meters long. You can also buy a rope with a



## Loading the summarization dataset

In [ ]:
data = {
    "train": load_dataset("abisee/cnn_dailymail", "3.0.0", split="train[:5000]"),
    "test": load_dataset("abisee/cnn_dailymail", "3.0.0", split="test[:500]")
}
print(data["train"].column_names)

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

['article', 'highlights', 'id']


## Preparing the finetuning data

In [ ]:
MAX_LENGTH = 2000

def generate_prompt(data_point):
    return f"<human>: Summarize the following article:\n{data_point['article']}\n<assistant>: {data_point['highlights']}"

def generate_and_tokenize_prompt(data_point):
    full_prompt = generate_prompt(data_point)+tokenizer.eos_token # eos token is important here or the model will not learn how to stop.
    tokenized_full_prompt = tokenizer(full_prompt, return_tensors='pt')
    labels = tokenized_full_prompt.input_ids.clone() # Create the labels by cloning input_ids

    prompt = full_prompt[:full_prompt.find("<assistant>")] + "<assistant>:"
    end_prompt_idx = len(tokenizer(prompt, return_tensors="pt").input_ids[0]) # Get the index of the '<assistant>:' (or the equivalent token) in order to replace all but response tokens with -100

    labels[:, :end_prompt_idx] = -100

    return {
        'input_ids': tokenized_full_prompt.input_ids.flatten(),
        'labels': labels.flatten(),
        'attention_mask': tokenized_full_prompt.attention_mask.flatten(),
    }

dataset = data["train"].shuffle(seed=42).map(generate_and_tokenize_prompt, remove_columns=data["train"].column_names)
dataset = dataset.filter(lambda example: len(example["input_ids"]) <= MAX_LENGTH)
dataset = dataset.train_test_split(test_size=50, seed=42)
train_data, eval_data = dataset["train"], dataset["test"]


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

## Fine tuning

In [ ]:
OUTPUT_DIR = "experiments"

training_args = TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-4,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_steps=100,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=0.01,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    per_device_eval_batch_size=1,
    save_total_limit=3,
    output_dir=OUTPUT_DIR,
    report_to="tensorboard",
)

trainer = Trainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

model.config.use_cache = False
print("Held-out loss before finetuning:", trainer.evaluate()["eval_loss"])
trainer.train()

Training Loss,Validation Loss,Step
No log,2.364655,0


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,909 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 11,272,192 of 1,247,086,592 (0.90% trained)


Held-out loss before finetuning: 2.364654779434204
Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
25,1.775273,1.719623
50,1.639692,1.702815
75,1.635114,1.651994
100,1.547914,1.644701


Unsloth: Restored added_tokens_decoder metadata in experiments/checkpoint-100/tokenizer_config.json.


TrainOutput(global_step=100, training_loss=1.6687324619293213, metrics={'train_runtime': 652.759, 'train_samples_per_second': 1.226, 'train_steps_per_second': 0.153, 'total_flos': 3930008535085056.0, 'train_loss': 1.6687324619293213, 'epoch': 0.16296598085149724})

## See the results

In [ ]:
%%time
device = "cuda:0"

sample = data["test"][0]

prompt_en = (
    "<human>: Summarize the following article:\n"
    f"{sample['article']}\n"
    "<assistant>:"
)

encoding = tokenizer(prompt_en, return_tensors="pt").to(device)

with torch.inference_mode():
    outputs = model.generate(
        input_ids=encoding.input_ids,
        attention_mask=encoding.attention_mask,
        generation_config=generation_config,
    )

print("Article summary expected:")
print(sample["highlights"])

print("\nGenerated summary:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("Held-out loss after finetuning:", trainer.evaluate()["eval_loss"])

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Article summary expected:
Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

Generated summary:
<human>: Summarize the following article:
(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC's founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war 

Training Loss,Validation Loss,Step
1.547914,1.644701,100


Held-out loss after finetuning: 1.644700527191162
CPU times: user 22.1 s, sys: 126 ms, total: 22.2 s
Wall time: 22.7 s
